<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [21]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [22]:
train_data = datasets.MNIST('.', download=True, train=True, transform=transform)
test_data = datasets.MNIST('.', download=True, train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000, shuffle=False)


In [23]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [24]:
epochs = 5
for epoch in range(1, epochs + 1):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 200 == 0:
            print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.4f}")

Epoch 1 [0/60000] Loss: 2.3122
Epoch 1 [12800/60000] Loss: 0.1879
Epoch 1 [25600/60000] Loss: 0.1479
Epoch 1 [38400/60000] Loss: 0.2909
Epoch 1 [51200/60000] Loss: 0.2683
Epoch 2 [0/60000] Loss: 0.2494
Epoch 2 [12800/60000] Loss: 0.0503
Epoch 2 [25600/60000] Loss: 0.1386
Epoch 2 [38400/60000] Loss: 0.0606
Epoch 2 [51200/60000] Loss: 0.0640
Epoch 3 [0/60000] Loss: 0.0567
Epoch 3 [12800/60000] Loss: 0.0720
Epoch 3 [25600/60000] Loss: 0.0913
Epoch 3 [38400/60000] Loss: 0.1814
Epoch 3 [51200/60000] Loss: 0.0721
Epoch 4 [0/60000] Loss: 0.0286
Epoch 4 [12800/60000] Loss: 0.0144
Epoch 4 [25600/60000] Loss: 0.0480
Epoch 4 [38400/60000] Loss: 0.0216
Epoch 4 [51200/60000] Loss: 0.0461
Epoch 5 [0/60000] Loss: 0.0285
Epoch 5 [12800/60000] Loss: 0.0169
Epoch 5 [25600/60000] Loss: 0.0789
Epoch 5 [38400/60000] Loss: 0.1779
Epoch 5 [51200/60000] Loss: 0.0120


In [25]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

print(f"Test Accuracy: {correct}/{total} ({100. * correct / total:.1f}%)")

Test Accuracy: 9727/10000 (97.3%)


In [26]:
torch.save(model.state_dict(), "mnist_model.pth")
print("Model saved")

Model saved
